In [0]:
import urllib.request
import json
from pyspark.sql.functions import from_json, col, schema_of_json, lit
from pyspark.sql.types import StringType

url = "https://randomuser.me/api/"

with urllib.request.urlopen(url) as response:
    data = str(response.read().decode("utf-8"))

raw_df = spark.createDataFrame([(data,)], "value STRING")
schema_str = spark.range(1).select(schema_of_json(lit(data)).alias("schema")).collect()[0]["schema"]
df = raw_df.select(from_json(col("value"), schema_str).alias("parsed")).select("parsed.*")

tar_df = df.selectExpr(
                       "info.page",
                       "info.results",
                       "info.seed",
                       "info.version",
                       "explode(results) AS result",
                       
                       
)
df.show()
tar_df.show()
df.printSchema()